# Episode 12 — Your first custom overlay

Until now we used the base overlay that came on the SD card. Now we build our own in Vivado
and load it from Python. It is deliberately tiny:

```
 ARM (PS) --M_AXI_GP0--> AXI interconnect --> AXI GPIO "gpio_led_sw"
                                                 channel 1 (4 outputs) --> LEDs
                                                 channel 2 (4 inputs)  <-- slide switches
```

## Building it in Vivado 2022.2

The folder for this episode has `first_overlay.tcl`, which does every step below. From a
command prompt in that folder (with Vivado's `bin` folder on the PATH):

```
vivado -mode batch -source first_overlay.tcl -tclargs xc7z020clg400-1 work overlays/z20
```

(use `xc7z010clg400-1` and `overlays/z10` for the Z7-10 board). It takes about 10 minutes.
By hand, the steps are:

1. New RTL project for the board's Zynq part.
2. **Create Block Design**, add **ZYNQ7 Processing System** with the EDGE ZYNQ settings
   (the EDGE board file preset), and run **Run Block Automation**.
3. Add **AXI GPIO**: enable dual channel; channel 1 *All Outputs*, width 4; channel 2 *All Inputs*, width 4.
4. **Run Connection Automation**; make the two GPIO interfaces external as `leds` and `sws`.
5. Add the pin constraints (`first_overlay.xdc`), create the HDL wrapper, **Generate Bitstream**.
6. Copy two files to the board, side by side and with the same name:
   * `first_overlay.bit`: the `first_overlay_wrapper.bit` from `<project>.runs/impl_1/`, renamed
   * `first_overlay.hwh` from `<project>.gen/sources_1/bd/first_overlay/hw_handoff/`

PYNQ reads the `.hwh` to learn what is inside the overlay; without it, the overlay loads but
Python cannot find the blocks.

## 1. Load the overlay

Loading a new overlay **replaces** the base overlay: the 7-segment display, buzzer and HDMI stop
until you load `base.bit` again.

In [ ]:
from pynq import Overlay
from pynq.overlays.base import board_variant
import time

chip = {"EDGE-Z10": "z10", "EDGE-Z20": "z20"}[board_variant()]
ol = Overlay("overlays/{}/first_overlay.bit".format(chip))

for name, ip in ol.ip_dict.items():
    if "phys_addr" not in ip:          # the processor itself has no address
        continue
    print("{:15s} 0x{:08X}  {}".format(name, ip["phys_addr"], ip["type"]))

## 2. PYNQ chose a driver for you

PYNQ matched the block's type (`xilinx.com:ip:axi_gpio:2.0`) to its `AxiGPIO` driver.
We only have to say how wide each channel is and which way it points.

In [ ]:
gpio = ol.gpio_led_sw
print(type(gpio).__name__)

leds = gpio.channel1
leds.setlength(4)
leds.setdirection("out")

switches = gpio.channel2
switches.setlength(4)
switches.setdirection("in")

In [ ]:
for n in range(16):
    leds[0:4].write(n)
    time.sleep(0.2)
leds[0:4].off()
print("Switches: {:04b}".format(switches.read()))

## 3. The same thing with MMIO

The AXI GPIO data sheet (PG144) lists its registers: channel 1 data at offset `0x00`,
channel 2 data at offset `0x08`. Here they are, used directly.

In [ ]:
gpio.write(0x00, 0b1001)                      # LEDs
print("Switches: {:04b}".format(gpio.read(0x08)))

## 4. Switches to LEDs, in the new overlay

In [ ]:
end = time.time() + 10
while time.time() < end:
    leds[0:4].write(switches.read())
    time.sleep(0.05)
leds[0:4].off()

## 5. Back to the base overlay

In [ ]:
from pynq.overlays.base import BaseOverlay
base = BaseOverlay("base.bit")
base.seven_seg.show_text("bASE")

## Try it yourself

In Vivado, add a second AXI GPIO for the two push buttons (pins W15 = left SW7 and V15 = right SW5, with pull-downs),
rebuild, and print the button value from Python.